# Observatorio de Obras Públicas · 01 Extracción (SECOP I + SECOP II)

Trae los contratos de **obra** reales de SECOP I y SECOP II, los une en una sola tabla sin duplicados y la guarda en `datos/obras.csv`.

**Para analizar otra región, solo cambia la celda 1.**

## 1 · Configuración (lo único que se cambia)

In [1]:
DEPARTAMENTO = "CAUCA"
MUNICIPIOS   = ["GUAPI", "TIMBIQUI", "LOPEZ DE MICAY"]   # sin tildes, en mayúsculas
ANIO_INICIO  = 2022
ANIO_FIN     = 2026

## 2 · Herramientas

In [2]:
%pip install -q duckdb

import os, unicodedata
import requests
import pandas as pd
import duckdb

FUENTES = {
    "SECOP II": "https://www.datos.gov.co/resource/jbjy-vk9h.json",
    "SECOP I":  "https://www.datos.gov.co/resource/f789-7hwg.json",
}
pd.options.display.float_format = "{:,.0f}".format
print("Listo.")


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


Listo.


## 3 · Funciones
SECOP I y SECOP II llaman distinto a las mismas columnas. `CAMPOS` lista los nombres posibles de cada dato y el código usa el que exista en cada fuente.

In [3]:
CAMPOS = {
    "id":          ["id_contrato", "uid", "numero_de_constancia", "numero_del_contrato"],
    "departamento":["departamento", "departamento_entidad", "departamento_ejecucion"],
    "municipio":   ["ciudad", "municipio_entidad", "municipio_ejecucion", "municipios_ejecucion"],
    "tipo":        ["tipo_de_contrato"],
    "fecha_firma": ["fecha_de_firma", "fecha_de_firma_del_contrato", "fecha_firma"],
    "objeto":      ["objeto_del_contrato", "descripcion_del_proceso", "detalle_del_objeto_a_contratar", "objeto_a_contratar"],
    "valor":       ["valor_del_contrato", "cuantia_contrato", "valor_contrato"],
    "valor_con_adiciones": ["valor_contrato_con_adiciones"],
    "entidad":     ["nombre_entidad"],
    "contratista": ["proveedor_adjudicado", "nom_razon_social_contratista"],
    "fecha_inicio":["fecha_de_inicio_del_contrato", "fecha_ini_ejec_contrato"],
    "fecha_fin":   ["fecha_de_fin_del_contrato", "fecha_fin_ejec_contrato"],
    "dias_adicionados": ["dias_adicionados"],
    "estado":      ["estado_contrato", "estado_del_proceso"],
    "url":         ["urlproceso", "ruta_proceso_en_secop_i"],
}


def normalizar(texto):
    """Quita tildes, espacios sobrantes y pasa a mayúsculas: 'Timbiquí ' -> 'TIMBIQUI'."""
    if texto is None or (isinstance(texto, float) and pd.isna(texto)):
        return ""
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.upper().split())


def columnas_de(url):
    """Nombres de las columnas reales de una fuente."""
    r = requests.get(url, params={"$limit": 1}, timeout=60)
    datos = r.json()
    if r.status_code != 200 or not isinstance(datos, list) or not datos:
        raise RuntimeError(f"La fuente {url} respondió {r.status_code}: {str(datos)[:300]}")
    return list(datos[0].keys())


def elegir(columnas):
    """Para cada dato de CAMPOS, la primera columna que exista en la fuente."""
    return {dato: next((c for c in opciones if c in columnas), None) for dato, opciones in CAMPOS.items()}


def consultar_api(url, where, tamano_pagina=5000):
    """Trae todos los registros que cumplen el filtro, por páginas."""
    filas, offset = [], 0
    while True:
        r = requests.get(url, params={"$where": where, "$limit": tamano_pagina,
                                      "$offset": offset, "$order": ":id"}, timeout=180)
        if r.status_code != 200:
            raise RuntimeError(f"La API respondió {r.status_code}: {r.text[:400]}")
        pagina = r.json()
        filas.extend(pagina)
        print(f"    {len(filas):,} registros...")
        if len(pagina) < tamano_pagina:
            return pd.DataFrame(filas)
        offset += tamano_pagina


def traer_fuente(nombre, url):
    print(f"\n{nombre}")
    col = elegir(columnas_de(url))
    faltan = [d for d in ("departamento", "municipio", "tipo", "fecha_firma") if not col[d]]
    if faltan:
        print(f"  No encontré columnas para {faltan}. Envíame este mensaje.")
        return pd.DataFrame()

    where = (f"upper({col['departamento']}) like '%{DEPARTAMENTO}%' "
             f"AND upper({col['tipo']}) like '%OBRA%'")
    try:   # primero intenta filtrar también por fecha en la API
        df = consultar_api(url, where + f" AND {col['fecha_firma']} >= '{ANIO_INICIO}-01-01T00:00:00'")
    except RuntimeError:
        print("  La fecha no se pudo filtrar en la API; se filtra después.")
        df = consultar_api(url, where)
    if df.empty:
        return df

    # Tabla con nombres comunes para las dos fuentes
    t = pd.DataFrame({dato: df[c] if c and c in df else None for dato, c in col.items()})
    t["fuente"] = nombre
    t["municipio"] = t["municipio"].map(normalizar)
    t["fecha_firma"] = pd.to_datetime(t["fecha_firma"], errors="coerce")
    for c in ("valor", "valor_con_adiciones", "dias_adicionados"):
        t[c] = pd.to_numeric(t[c], errors="coerce")

    objetivo = [normalizar(m) for m in MUNICIPIOS]
    t = t[t["municipio"].apply(lambda m: any(o in m for o in objetivo))]
    t = t[t["fecha_firma"].dt.year.between(ANIO_INICIO, ANIO_FIN)]
    # Deja el nombre limpio del municipio (ej. 'LOPEZ DE MICAY')
    t["municipio"] = t["municipio"].apply(lambda m: next(o for o in objetivo if o in m))
    print(f"  -> {len(t):,} contratos de obra en los municipios elegidos")
    return t

## 4 · Traer y unir SECOP I y SECOP II

In [4]:
tablas = [traer_fuente(n, u) for n, u in FUENTES.items()]
obras = pd.concat([t for t in tablas if not t.empty], ignore_index=True)

antes = len(obras)
obras = obras.drop_duplicates(subset=["id", "fuente"]).reset_index(drop=True)
print(f"\nTotal: {len(obras):,} contratos ({antes - len(obras)} duplicados eliminados)")
print(obras.groupby(["fuente", "municipio"]).size())


SECOP II


    3,192 registros...
  -> 6 contratos de obra en los municipios elegidos

SECOP I


    3,882 registros...
  -> 359 contratos de obra en los municipios elegidos

Total: 363 contratos (2 duplicados eliminados)
fuente    municipio     
SECOP I   GUAPI              72
          LOPEZ DE MICAY    275
          TIMBIQUI           10
SECOP II  GUAPI               1
          TIMBIQUI            5
dtype: int64


## 5 · Primera consulta SQL: obras y valor por municipio y año
Primer vistazo. Todavía falta la limpieza (paso 02).

In [5]:
duckdb.sql("""
    SELECT municipio,
           year(fecha_firma)  AS anio,
           COUNT(*)           AS obras,
           SUM(valor)         AS valor_contratado
    FROM obras
    GROUP BY municipio, anio
    ORDER BY municipio, anio
""").df()

,municipio,anio,obras,valor_contratado
0,GUAPI,2022,5,"7,564,121,120"
1,GUAPI,2023,2,"543,546,378"
2,GUAPI,2024,15,"813,420,312"
3,GUAPI,2025,31,"7,145,046,246"
4,GUAPI,2026,20,"12,045,714,838"
5,LOPEZ DE MICAY,2022,85,"14,240,683,314"
6,LOPEZ DE MICAY,2023,63,"1,953,458,447"
7,LOPEZ DE MICAY,2024,78,"3,203,837,143"
8,LOPEZ DE MICAY,2025,37,"7,542,614,847"
9,LOPEZ DE MICAY,2026,12,"2,627,137,031"


## 6 · Guardar

In [6]:
os.makedirs("datos", exist_ok=True)
obras.to_csv("datos/obras.csv", index=False)
print("Guardado: datos/obras.csv")

try:   # si el código corre en un servidor de Colab, descarga el archivo
    from google.colab import files
    files.download("datos/obras.csv")
except Exception:
    pass

Guardado: datos/obras.csv
